# Held-input FFT rounding with NumPy and JAX

Run this notebook with NumPy, JAX and Jupyter installed. It requires no PyCBC,
LALSuite, data files or other notebook. Fixed inputs separate forward and inverse
transform differences from preceding calculations. Each table states the NumPy
FFT return dtype and the explicit output cast, with matching normalization.

Recorded results belong to the displayed libraries and device. A differing
output identifies a transform boundary; it does not identify a vendor's internal
butterfly, accumulator precision or instruction. Zero differences are valid.
The direct DFT below is a double-precision mathematical diagnostic, not a
replacement for the original PyCBC FFT provider. For original-provider validation,
see [the PyCBC comparison notebook](jax_fft_numerical_differences.ipynb).

In [1]:
import math
import platform
import sys
import numpy as np
import jax
import jax.numpy as jnp
import jaxlib

# Enable double precision before creating any JAX array. A single-precision
# input nevertheless retains single-precision FFT output.
jax.config.update("jax_enable_x64", True)
device = jax.devices()[0]
print({"Python": sys.version.split()[0], "NumPy": np.__version__,
       "JAX": jax.__version__, "jaxlib": jaxlib.__version__,
       "architecture": platform.machine(), "platform": device.platform,
       "device kind": device.device_kind, "x64": jax.config.jax_enable_x64})

{'Python': '3.13.3', 'NumPy': '2.2.5', 'JAX': '0.11.2', 'jaxlib': '0.11.2', 'architecture': 'x86_64', 'platform': 'gpu', 'device kind': 'NVIDIA GeForce RTX 3090', 'x64': True}


## Inputs and comparison units

Lengths 30 and 32 contrast a non-power of two with a power of two. Forward
inputs and inverse spectra are generated independently with a fixed seed. Real
inverse spectra have real DC and Nyquist bins. Both libraries use `norm="backward"`:
forward transforms are unscaled and inverse transforms include `1/N`.

The output cast is explicit in both paths. A bin mismatch counts differing
stored bytes, including signed zero. Absolute error is evaluated after widening
stored outputs to double precision. The maximum component ULP distance is the
number of adjacent IEEE float representations separating any real or imaginary
component; near zero it can be large even when absolute error is tiny. The
component distance counts the two signed-zero representations separately.

In [2]:
def difference(actual, expected):
    actual, expected = np.ascontiguousarray(actual), np.ascontiguousarray(expected)
    assert actual.dtype == expected.dtype and actual.shape == expected.shape
    item_bytes = actual.dtype.itemsize
    byte_rows_a = actual.view(np.uint8).reshape(-1, item_bytes)
    byte_rows_b = expected.view(np.uint8).reshape(-1, item_bytes)
    bins = int(np.count_nonzero(np.any(byte_rows_a != byte_rows_b, axis=1)))
    wide_dtype = np.complex128 if np.iscomplexobj(actual) else np.float64
    absolute = float(np.max(np.abs(actual.astype(wide_dtype)
                                   - expected.astype(wide_dtype))))
    real_dtype = actual.real.dtype
    components_a = actual.view(real_dtype).ravel()
    components_b = expected.view(real_dtype).ravel()
    unsigned_dtype = np.uint32 if real_dtype.itemsize == 4 else np.uint64
    sign = 1 << (8 * real_dtype.itemsize - 1)
    mask = 2 * sign - 1
    # Python integers avoid uint64 overflow when computing representation gaps.
    def ordered(bits):
        bits = int(bits)
        return (~bits & mask) if bits & sign else bits | sign
    max_ulps = max(abs(ordered(a) - ordered(b)) for a, b in
                   zip(components_a.view(unsigned_dtype), components_b.view(unsigned_dtype)))
    return bins, absolute, max_ulps

rng = np.random.default_rng(19073)
cases = []
for n in (30, 32):
    for input_dtype in (np.float32, np.float64, np.complex64, np.complex128):
        is_real = np.issubdtype(input_dtype, np.floating)
        real_dtype = np.float32 if input_dtype in (np.float32, np.complex64) else np.float64
        complex_dtype = np.complex64 if real_dtype == np.float32 else np.complex128
        forward = rng.standard_normal(n)
        if not is_real:
            forward = forward + 1j * rng.standard_normal(n)
        forward = forward.astype(input_dtype)
        count = n // 2 + 1 if is_real else n
        inverse = (rng.standard_normal(count)
                   + 1j * rng.standard_normal(count)).astype(complex_dtype)
        if is_real:
            inverse[0] = inverse[0].real
            inverse[-1] = inverse[-1].real  # Both lengths here are even.
        cases.append(dict(n=n, name=np.dtype(input_dtype).name, real=is_real,
                          real_dtype=real_dtype, complex_dtype=complex_dtype,
                          forward=forward, inverse=inverse))

## Hold forward and inverse inputs separately

The inverse table uses its independently stored spectrum, rather than a JAX
forward result. Each row therefore measures only its named transform. NumPy's
return dtype is observed at runtime: do not assume every NumPy version promotes
complex64 transforms to complex128. The cast column is the common storage dtype,
not a claim about either library's undocumented internal arithmetic.

In [3]:
print("direction input       N  NumPy return cast         byte bins  max abs       component ULPs")
for case in cases:
    for direction in ('forward', 'inverse'):
        values = case[direction]
        if direction == 'forward':
            np_transform = np.fft.rfft if case['real'] else np.fft.fft
            jax_transform = jnp.fft.rfft if case['real'] else jnp.fft.fft
            output_dtype = case['complex_dtype']
        else:
            np_transform = np.fft.irfft if case['real'] else np.fft.ifft
            jax_transform = jnp.fft.irfft if case['real'] else jnp.fft.ifft
            output_dtype = case['real_dtype'] if case['real'] else case['complex_dtype']
        numpy_api = np_transform(values, n=case['n'], norm='backward')
        numpy_stored = np.asarray(numpy_api, dtype=output_dtype)
        jax_stored = np.asarray(jax_transform(jax.device_put(values, device),
                                n=case['n'], norm='backward'), dtype=output_dtype)
        assert np.isfinite(numpy_stored).all() and np.isfinite(jax_stored).all()
        bins, absolute, ulps = difference(jax_stored, numpy_stored)
        print(f"{direction:9} {case['name']:10} {case['n']:2} {numpy_api.dtype.name:12} "
              f"{numpy_stored.dtype.name:12} {bins:3}/{numpy_stored.size:<3} "
              f"{absolute:12.5g} {ulps:14}")

direction input       N  NumPy return cast         byte bins  max abs       component ULPs
forward   float32    30 complex64    complex64     15/16     9.611e-07      880803840
inverse   float32    30 float32      float32       16/30    5.9605e-08             34
forward   float64    30 complex128   complex128    15/16    3.5527e-15 4373558188130172928
inverse   float64    30 float64      float64       18/30    1.1796e-16             60
forward   complex64  30 complex64    complex64     29/30    1.7193e-06             54
inverse   complex64  30 complex64    complex64     28/30     6.664e-08            546
forward   complex128 30 complex128   complex128    28/30    2.5895e-15             64
inverse   complex128 30 complex128   complex128    28/30    1.1102e-16             69
forward   float32    32 complex64    complex64     16/17     9.611e-07             21
inverse   float32    32 float32      float32       15/32    5.9605e-08             40
forward   float64    32 complex128   complex

## Separate the inverse normalization boundary

An unnormalized inverse can be written as `N * inverse(norm="backward")` or
requested through `norm="forward"`. These expressions agree in real arithmetic.
Here the first path casts the normalized result to the stated storage precision
**before** multiplying by `N` in that real precision. The second path requests the
library's unnormalized result and then casts. Their differing-bin counts measure
this placement within each library; they need not differ on every device.
Powers of two have exact binary scale factors, but that alone does not guarantee
that different FFT calls use the same internal ordering.

In [4]:
print("input       N  cast         NumPy scale paths  JAX scale paths  same-path JAX/NumPy max abs")
for case in cases:
    output_dtype = case['real_dtype'] if case['real'] else case['complex_dtype']
    np_transform = np.fft.irfft if case['real'] else np.fft.ifft
    jax_transform = jnp.fft.irfft if case['real'] else jnp.fft.ifft
    n = case['n']
    scale = np.asarray(n, dtype=case['real_dtype'])
    np_after = np.asarray(np_transform(case['inverse'], n=n, norm='backward'),
                          dtype=output_dtype) * scale
    np_requested = np.asarray(np_transform(case['inverse'], n=n, norm='forward'),
                              dtype=output_dtype)
    values_device = jax.device_put(case['inverse'], device)
    jax_after = np.asarray(jax_transform(values_device, n=n, norm='backward').astype(output_dtype)
                           * jnp.asarray(n, dtype=case['real_dtype']))
    jax_requested = np.asarray(jax_transform(values_device, n=n, norm='forward'),
                               dtype=output_dtype)
    np_bins = difference(np_after, np_requested)[0]
    jax_bins = difference(jax_after, jax_requested)[0]
    _, absolute, _ = difference(jax_after, np_after)
    print(f"{case['name']:10} {n:2} {np.dtype(output_dtype).name:12} "
          f"{np_bins:3}/{np_after.size:<3}            {jax_bins:3}/{jax_after.size:<3} "
          f"{absolute:20.5g}")

input       N  cast         NumPy scale paths  JAX scale paths  same-path JAX/NumPy max abs
float32    30 float32       25/30               0/30            1.9073e-06
float64    30 float64        2/30               0/30            3.5527e-15
complex64  30 complex64     29/30               0/30            2.1325e-06
complex128 30 complex128    11/30               0/30            3.5527e-15
float32    32 float32       20/32               0/32            1.9073e-06
float64    32 float64        0/32               0/32            3.5527e-15
complex64  32 complex64     28/32               0/32            1.9222e-06
complex128 32 complex128     0/32               0/32            3.2024e-15


## A small direct DFT as a mathematical diagnostic

The next calculation uses the defining complex128 exponential matrix and
matrix-vector product for lengths 7 and 8. It avoids an FFT decomposition, but its
phase evaluation and sums still round in double precision. It is not an exact
real-arithmetic answer and is not the agreed original backend. Comparing stored
FFT outputs with it can diagnose scale or precision mistakes; greater proximity
to this diagnostic does not replace a failed original-backend equality check.
The forward and inverse inputs are again independent.

In [5]:
def direct_dft(values, inverse=False):
    values = np.asarray(values, dtype=np.complex128)
    n = values.size
    indices = np.arange(n, dtype=np.float64)
    sign = 1.0 if inverse else -1.0
    phases = np.exp(sign * 2j * np.pi * np.outer(indices, indices) / n)
    result = phases @ values
    return result / n if inverse else result

print("direction input       N  NumPy max abs vs direct DFT  JAX max abs vs direct DFT")
for n in (7, 8):
    for input_dtype in (np.complex64, np.complex128):
        forward = (rng.standard_normal(n) + 1j * rng.standard_normal(n)).astype(input_dtype)
        inverse = (rng.standard_normal(n) + 1j * rng.standard_normal(n)).astype(input_dtype)
        recovered = direct_dft(direct_dft(forward), inverse=True)
        assert np.allclose(recovered, forward.astype(np.complex128), rtol=5e-14, atol=5e-14)
        for direction, values in (('forward', forward), ('inverse', inverse)):
            is_inverse = direction == 'inverse'
            oracle = direct_dft(values, inverse=is_inverse)
            np_transform = np.fft.ifft if is_inverse else np.fft.fft
            jax_transform = jnp.fft.ifft if is_inverse else jnp.fft.fft
            np_stored = np.asarray(np_transform(values, norm='backward'), dtype=input_dtype)
            jax_stored = np.asarray(jax_transform(jax.device_put(values, device),
                                    norm='backward'), dtype=input_dtype)
            np_error = np.max(np.abs(np_stored.astype(np.complex128) - oracle))
            jax_error = np.max(np.abs(jax_stored.astype(np.complex128) - oracle))
            print(f"{direction:9} {np.dtype(input_dtype).name:10} {n:2} "
                  f"{np_error:27.5g} {jax_error:26.5g}")

direction input       N  NumPy max abs vs direct DFT  JAX max abs vs direct DFT
forward   complex64   7                  1.2103e-07                 2.9766e-07
inverse   complex64   7                  1.1923e-07                 1.1923e-07
forward   complex128  7                  8.1886e-15                 8.7925e-15
inverse   complex128  7                  9.7996e-16                 9.7303e-16
forward   complex64   8                  2.1187e-07                 6.1205e-07
inverse   complex64   8                  2.7297e-08                 3.6029e-08
forward   complex128  8                  7.5886e-15                 7.9441e-15
inverse   complex128  8                  1.3369e-15                 1.2959e-15


## Show an addition-grouping mechanism explicitly

The zero-frequency DFT is a sum. These stored values have mathematical sum two.
Rounding after every left-to-right addition gives one, while summing two pairs
first gives zero. `math.fsum` supplies an independent scalar diagnostic here.
The NumPy/JAX zero bins are measured separately. This example proves that the
visible addition order matters; it does **not** assert that either FFT library
uses either displayed grouping internally.

In [6]:
print("input      sequential  paired  fsum  NumPy zero bin  JAX zero bin")
for real_dtype, large in ((np.float32, 1e8), (np.float64, 1e16)):
    terms = np.array([large, 1, -large, 1], dtype=real_dtype)
    sequential = real_dtype(0)
    for term in terms:
        sequential = real_dtype(sequential + term)
    paired = real_dtype(real_dtype(terms[0] + terms[1])
                        + real_dtype(terms[2] + terms[3]))
    exact_scalar = math.fsum(float(value) for value in terms)
    assert sequential == 1 and paired == 0 and exact_scalar == 2
    np_zero = np.fft.fft(terms, norm='backward')[0]
    jax_zero = np.asarray(jnp.fft.fft(jax.device_put(terms, device), norm='backward'))[0]
    print(f"{np.dtype(real_dtype).name:10} {float(sequential):10g} {float(paired):7g} "
          f"{exact_scalar:5g} {np_zero!s:15} {jax_zero!s}")

input      sequential  paired  fsum  NumPy zero bin  JAX zero bin
float32             1       0     2 (2+0j)          (2+0j)
float64             1       0     2 (2+0j)          (2+0j)


## Read the results within their scope

The tables keep inputs, output storage and transform normalization fixed. The
scale and addition controls expose visible rounding placements. A same-input
FFT discrepancy remains attributed to the installed FFT implementations unless
further evidence isolates their internal operations. These examples do not test
PyCBC dispatch, original FFTW planning, series metadata or full-search parity;
those checks remain in the [PyCBC comparison notebook](jax_fft_numerical_differences.ipynb).